# Hierarchical Bayesian Regression with PyMC: When Groups Share Strength

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/hierarchical_bayesian_regression.ipynb)

This notebook accompanies the blog post on [sesen.ai](https://sesen.ai/blog/hierarchical-bayesian-regression-pymc).

We build a hierarchical Bayesian regression model in PyMC to estimate insurance claim severity across policy types, demonstrating the power of partial pooling and shrinkage.

In [ ]:
# Install dependencies (Colab only)
!pip install -q pymc arviz

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

np.random.seed(42)
print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 1. Generate Synthetic Insurance Data

We create claims for three policy types with deliberately unbalanced sample sizes:
- **Auto**: 500 claims (plenty of data)
- **Home**: 300 claims (moderate data)
- **Commercial**: 50 claims (scarce data — this is where hierarchical models shine)

In [ ]:
# Three policy types with different sample sizes and true parameters
groups = {
    'Auto':       {'n': 500, 'intercept': 7.5, 'slope': 0.30},
    'Home':       {'n': 300, 'intercept': 8.2, 'slope': 0.50},
    'Commercial': {'n':  50, 'intercept': 9.0, 'slope': 0.70},
}

records = []
for i, (name, p) in enumerate(groups.items()):
    x = np.random.normal(12, 1.5, p['n'])  # log property value (~$160k median)
    noise = np.random.normal(0, 0.8, p['n'])
    y = p['intercept'] + p['slope'] * (x - 12) + noise
    for j in range(p['n']):
        records.append({
            'policy_type': name, 'group_idx': i,
            'log_property_value': x[j], 'log_claim_severity': y[j],
        })

df = pd.DataFrame(records)
policy_names = ['Auto', 'Home', 'Commercial']
n_types = len(policy_names)
idx = df['group_idx'].values
x_centered = df['log_property_value'].values - 12
y_data = df['log_claim_severity'].values

print(f"Total claims: {len(df)}")
print(df.groupby('policy_type').size())

In [ ]:
# Visualise the data
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(policy_names):
    mask = df['group_idx'] == i
    n = mask.sum()
    ax.scatter(df.loc[mask, 'log_property_value'],
               df.loc[mask, 'log_claim_severity'],
               alpha=0.4, s=20, c=colors[i], label=f'{name} (n={n})')
ax.set_xlabel('Log Property Value', fontsize=12)
ax.set_ylabel('Log Claim Severity', fontsize=12)
ax.set_title('Insurance Claims by Policy Type', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 2. The Three Pooling Strategies

We compare three approaches to modelling grouped data:

1. **Complete pooling**: One set of parameters for all groups (ignores group structure)
2. **No pooling**: Separate parameters per group (no information sharing)
3. **Partial pooling**: Hierarchical model (groups share a population distribution)

In [ ]:
# Model 1: Complete Pooling — one line for everyone
with pm.Model() as pooled_model:
    alpha = pm.Normal('alpha', mu=8, sigma=5)
    beta = pm.Normal('beta', mu=0, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=2)

    mu = alpha + beta * x_centered
    pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_data)

    pooled_trace = pm.sample(1000, tune=1500, cores=4, chains=4, random_seed=42)

print(az.summary(pooled_trace, var_names=['alpha', 'beta', 'sigma']))

In [ ]:
# Model 2: No Pooling — separate parameters per group
with pm.Model() as unpooled_model:
    alpha = pm.Normal('alpha', mu=8, sigma=5, shape=n_types)
    beta = pm.Normal('beta', mu=0, sigma=5, shape=n_types)
    sigma = pm.HalfNormal('sigma', sigma=2)

    mu = alpha[idx] + beta[idx] * x_centered
    pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_data)

    unpooled_trace = pm.sample(1000, tune=1500, cores=4, chains=4, random_seed=42)

print(az.summary(unpooled_trace, var_names=['alpha', 'beta', 'sigma']))

In [ ]:
# Model 3: Partial Pooling (Hierarchical) — groups share a population distribution
with pm.Model() as hierarchical_model:
    # Hyperpriors: the population distribution
    mu_alpha = pm.Normal('mu_alpha', mu=8, sigma=2)
    sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=2)
    mu_beta = pm.Normal('mu_beta', mu=0, sigma=2)
    sigma_beta = pm.HalfNormal('sigma_beta', sigma=2)

    # Group-level parameters, drawn from the population
    alpha = pm.Normal('alpha', mu=mu_alpha, sigma=sigma_alpha, shape=n_types)
    beta = pm.Normal('beta', mu=mu_beta, sigma=sigma_beta, shape=n_types)

    # Observation noise
    sigma = pm.HalfNormal('sigma', sigma=2)

    # Linear model
    mu = alpha[idx] + beta[idx] * x_centered
    pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_data)

    # Sample the posterior
    hierarchical_trace = pm.sample(
        1000, tune=1500, cores=4, chains=4,
        random_seed=42, target_accept=0.9,
    )

print(az.summary(hierarchical_trace,
      var_names=['alpha', 'beta', 'sigma', 'mu_alpha', 'sigma_alpha']))

## 3. Comparing the Three Approaches

In [ ]:
# Compare regression lines from all three models
x_grid = np.linspace(df['log_property_value'].min(), df['log_property_value'].max(), 100)
x_grid_c = x_grid - 12

pooled_summary = az.summary(pooled_trace, var_names=['alpha', 'beta'])
unpooled_summary = az.summary(unpooled_trace, var_names=['alpha', 'beta'])
hier_summary = az.summary(hierarchical_trace, var_names=['alpha', 'beta'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

pooled_a = float(pooled_summary.loc['alpha', 'mean'])
pooled_b = float(pooled_summary.loc['beta', 'mean'])

for panel, (ax, title, trace_type) in enumerate(zip(
    axes,
    ['Complete Pooling', 'No Pooling', 'Partial Pooling (Hierarchical)'],
    ['pooled', 'unpooled', 'hierarchical']
)):
    for i, name in enumerate(policy_names):
        mask = df['group_idx'] == i
        ax.scatter(df.loc[mask, 'log_property_value'],
                   df.loc[mask, 'log_claim_severity'],
                   alpha=0.2, s=12, c=colors[i])

    if trace_type == 'pooled':
        y_line = pooled_a + pooled_b * x_grid_c
        ax.plot(x_grid, y_line, 'k-', lw=2, label='Pooled fit')
        ax.legend(fontsize=9)
    else:
        summary = unpooled_summary if trace_type == 'unpooled' else hier_summary
        for i, name in enumerate(policy_names):
            a_i = float(summary.loc[f'alpha[{i}]', 'mean'])
            b_i = float(summary.loc[f'beta[{i}]', 'mean'])
            y_line = a_i + b_i * x_grid_c
            ax.plot(x_grid, y_line, color=colors[i], lw=2, label=name)
        ax.legend(fontsize=9)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Log Property Value', fontsize=10)
    if panel == 0:
        ax.set_ylabel('Log Claim Severity', fontsize=10)

fig.suptitle('Three Pooling Strategies Compared', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Shrinkage: The Key Insight

Shrinkage is the defining feature of hierarchical models. Groups with less data get "pulled" more
toward the population mean. Let's visualise this effect on the intercept estimates.

In [ ]:
# Compare raw group means vs hierarchical posterior means
raw_means = [df.loc[df['group_idx'] == i, 'log_claim_severity'].mean() for i in range(n_types)]
hier_means = [float(hier_summary.loc[f'alpha[{i}]', 'mean']) for i in range(n_types)]
pop_mean = float(az.summary(hierarchical_trace, var_names=['mu_alpha'])['mean'].iloc[0])

fig, ax = plt.subplots(figsize=(8, 5))
for i, name in enumerate(policy_names):
    n_i = groups[name]['n']
    ax.scatter(raw_means[i], i, marker='o', s=120, c=colors[i],
               edgecolors='black', zorder=5, label='Raw mean' if i == 0 else None)
    ax.scatter(hier_means[i], i, marker='^', s=120, c=colors[i],
               edgecolors='black', zorder=5, label='Hierarchical mean' if i == 0 else None)
    ax.annotate('', xy=(hier_means[i], i), xytext=(raw_means[i], i),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2))
    ax.text(max(raw_means[i], hier_means[i]) + 0.05, i + 0.12,
            f'{name} (n={n_i})', fontsize=11, color=colors[i], fontweight='bold')

ax.axvline(pop_mean, color='grey', linestyle='--', lw=1.5, alpha=0.7,
           label=f'Population mean ({pop_mean:.2f})')
ax.set_yticks(range(n_types))
ax.set_yticklabels([''] * n_types)
ax.set_xlabel('Intercept Estimate', fontsize=12)
ax.set_title('Shrinkage: Raw Group Means vs Hierarchical Posterior Means', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nCommercial shrinkage: raw mean {raw_means[2]:.3f} -> posterior {hier_means[2]:.3f}")
print(f"Auto shrinkage:       raw mean {raw_means[0]:.3f} -> posterior {hier_means[0]:.3f}")

## 5. MCMC Diagnostics

Before trusting the results, we check that the sampler converged properly.

In [ ]:
# Trace plots
az.plot_trace(hierarchical_trace,
              var_names=['alpha', 'beta', 'sigma'],
              compact=True, figsize=(14, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Check R-hat, ESS, and divergences
summary = az.summary(hierarchical_trace,
                     var_names=['alpha', 'beta', 'sigma', 'mu_alpha',
                                'sigma_alpha', 'mu_beta', 'sigma_beta'])
print(summary[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

## 6. Going Deeper: Laplace Likelihood and Heteroscedastic Models

Insurance claims are heavy-tailed: most claims are small, but a few are enormous.
A Laplace likelihood handles this better than a Normal. We can also model the
**spread** (scale parameter) as a function of covariates — this is heteroscedastic regression.

In [ ]:
# Heteroscedastic hierarchical model with Laplace likelihood
# This matches the structure of the original production code

with pm.Model() as laplace_model:
    # Hyperpriors for location
    mu_alpha = pm.Normal('mu_alpha', mu=0, sigma=1)
    sigma_alpha = pm.InverseGamma('sigma_alpha', alpha=2, beta=5)
    mu_beta = pm.Normal('mu_beta', mu=0, sigma=1)
    sigma_beta = pm.InverseGamma('sigma_beta', alpha=2, beta=5)

    # Group-level location parameters
    alpha = pm.Normal('alpha', mu=mu_alpha, sigma=sigma_alpha, shape=n_types)
    beta = pm.Normal('beta', mu=mu_beta, sigma=sigma_beta, shape=n_types)

    # Hyperpriors for scale
    mu_gamma = pm.Normal('mu_gamma', mu=0, sigma=1)
    sigma_gamma = pm.InverseGamma('sigma_gamma', alpha=2, beta=5)

    # Group-level scale parameters
    gamma = pm.Normal('gamma', mu=mu_gamma, sigma=sigma_gamma, shape=n_types)

    # Location: exp(alpha + beta * x) ensures positivity
    mu = pm.math.exp(alpha[idx] + beta[idx] * x_centered)

    # Scale: exp(gamma) ensures positivity
    b = pm.math.exp(gamma[idx])

    # Laplace likelihood (heavier tails than Normal)
    pm.Laplace('y_obs', mu=mu, b=b, observed=np.exp(y_data))

    laplace_trace = pm.sample(1000, tune=2000, cores=4, chains=4,
                              random_seed=42, target_accept=0.9)

print(az.summary(laplace_trace, var_names=['alpha', 'beta', 'gamma']))

## 7. Exercises

1. **Add a fourth policy type** with only 10 data points and a very different intercept (e.g., 11.0). How much does the hierarchical model shrink it compared to the other groups?

2. **Non-centred parameterisation**: Rewrite the hierarchical model using `alpha = mu_alpha + sigma_alpha * alpha_offset` where `alpha_offset ~ Normal(0, 1)`. Does this reduce divergences?

3. **Model comparison**: Use `az.compare()` with WAIC or LOO to formally compare the pooled, unpooled, and hierarchical models. Which one has the best predictive performance?

4. **Posterior predictive checks**: Generate posterior predictive samples and compare them against the observed data distribution for each policy type. Does the model capture the group-level differences?

5. **Three-tier hierarchy**: Extend the model to include a second grouping variable (e.g., region: urban/suburban). Each (region, policy_type) combination should have its own parameters, with policy-type-level hyperpriors, and population-level hyper-hyperpriors.